# 07: PV/PVC, ConfigMap & Secrets — Full Walkthrough

**Stack:** Frontend (FastAPI) → Backend (FastAPI + psycopg2) → PostgreSQL (StatefulSet + PVC)

**Pattern:** Mock data first → real database later. API contract stays unchanged.

## Architecture

```
Browser
   │
   ▼
frontend-service (ClusterIP)          ──→  Frontend Pods ×3
   │                                           │ HTTP GET /users
   ▼                                           ▼
backend-service (ClusterIP)           ──→  FastAPI Pods ×3
   │                                           │ env vars from ConfigMap + Secret
   │                                           │ psycopg2 connection
   ▼                                           ▼
postgres-service (headless)           ──→  PostgreSQL StatefulSet (postgres-0)
                                                │
                                                ▼
                                             PVC (postgres-storage-postgres-0)
                                                │
                                                ▼
                                             PV (kind host storage)
```

## 1. Cluster Setup

```bash
kind create cluster --name k8-lab
kubectl create namespace dev
```

> **❌ Don't do:** `kind create namespace dev` — `kind` doesn't create namespaces. Use `kubectl`.

## 2. Frontend

**Files:** `frontend/main.py`, `frontend/Dockerfile`, `frontend/requirements.txt`, `frontend/frontend.yaml`

**Commands:**
```bash
cd 07.pv,configmap,secrers/frontend
docker build -t frontend-07:1.0 .
kind load docker-image frontend-07:1.0 --name k8-lab
kubectl apply -f frontend.yaml
kubectl get pods -n dev   # 3/3 Running
```

> **❌ Mistakes made:**
> - `docker build` from root — no Dockerfile there
> - `kind create namespace dev` — kind can't do that
> - `kubectl get nods` — typo, should be `nodes`

## 3. Backend v1.0 — Mock JSON

```python
# backend/main.py — hardcoded mock data
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    users = [
        {"id": 1, "name": "Alice"},
        {"id": 2, "name": "Bob"},
        {"id": 3, "name": "Charlie"}
    ]
    return {"users": users}
```

```bash
cd 07.pv,configmap,secrers/backend
docker build -t backend-07:1.0 .
kind load docker-image backend-07:1.0 --name k8-lab
kubectl apply -f backend.yaml
kubectl port-forward service/backend 8080:80 -n dev
# → {"users":[{"id":1,"name":"Alice"},...]}
```

**Pattern:** Mock data first → real DB later. API contract stays unchanged.

## 4. ConfigMap & Secret — Theory

### ConfigMap
- Stores non-sensitive config: `DB_HOST=postgres`, `LOG_LEVEL=INFO`
- Usually safe to commit to Git

### Secret
- Stores sensitive data: passwords, API keys, tokens
- **Base64 is encoding, NOT encryption** — instant decode:
  ```bash
  echo "bXlwYXNzd29yZA==" | base64 -d  # → mypassword
  ```
- Never commit real Secret values to Git

### Production Pattern
```
GitHub → CI/CD → Secrets Manager → K8s Secret (never in repo)
```
Commit `secret.example.yaml` (template) instead of real `secret.yaml`.

### Rule of Thumb
| Object | Content | Commit? |
|--------|---------|---------|
| ConfigMap | Non-sensitive settings | Usually yes |
| Secret | Passwords, keys, tokens | Never real values |
| `secret.example.yaml` | Placeholder template | Yes |
| Actual Secret | Real credentials | No — deploy-time only |

**For this project:** `password123` is fine — local kind cluster, learning only. Final project will use a secrets manager.

## 5. PostgreSQL — Persistent Storage

### 5a. Secret (`postgres/secrets.yaml`)
Stores `POSTGRES_USER: admin`, `POSTGRES_PASSWORD: password123`. Injected via `secretKeyRef`.

### 5b. PVC (`postgres/pvc.yaml`)
Requests 1Gi `ReadWriteOnce` storage. Initially `Pending` until bound by a Pod.

### 5c. StatefulSet (`postgres/statefulset.yaml`)

> **❌ CRITICAL ERROR:**
> ```
> Error: unknown field "spec.template.spec.containers[0].volumes"
> ```
> **Why?** In StatefulSet, persistent storage uses `volumeClaimTemplates` at `spec` level — **NOT** `volumes:` inside the container spec (that's a Deployment thing).

**Key StatefulSet features:**
- Pods named `postgres-0`, `postgres-1`, etc. (stable identity)
- Each Pod gets its own PVC automatically via `volumeClaimTemplates`
- Ordered creation and termination
- Works with headless service for direct Pod DNS

### 5d. Headless Service (`postgres/service.yaml`)
`clusterIP: None` → enables `postgres-0.postgres.dev.svc.cluster.local` DNS.

### Commands
```bash
kubectl apply -f postgres/secrets.yaml
kubectl apply -f postgres/pvc.yaml
kubectl apply -f postgres/statefulset.yaml   # ❌ error first → fixed → ✅
kubectl apply -f postgres/service.yaml
kubectl get pods -n dev   # postgres-0 should be Running
```

## 6. Backend v1.1 — Real PostgreSQL

**What changed from v1.0:**

| Aspect | v1.0 (Mock) | v1.1 (Real DB) |
|--------|-------------|----------------|
| Data source | Hardcoded JSON | PostgreSQL `users` table |
| Dependencies | fastapi, uvicorn | + psycopg2-binary |
| DB config | None | `os.getenv()` from ConfigMap + Secret |

```python
from fastapi import FastAPI
import psycopg2
import os

app = FastAPI()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT", "5432")
)

@app.get("/")
def get_users():
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM users")
    rows = cursor.fetchall()
    cursor.close()
    return {"users": rows}
```

**Key principle:** Backend reads env vars — never knows if they came from ConfigMap, Secret, or elsewhere.

### Rolling Update
```bash
docker build -t backend-07:1.1 .
kind load docker-image backend-07:1.1 --name k8-lab
kubectl apply -f backend.yaml
kubectl rollout status deployment/backend-deployment -n dev
```

```
Old Pod (v1.0)  →  New Pod (v1.1) starts  →  Old Pod drains & stops
                        ↓
                Repeat for all 3 replicas
```

- Zero-downtime: new Pods start before old ones terminate
- `kubectl rollout status` tracks progress
- Backend never stops serving

## Key Takeaways

| Concept | What We Learned |
|---------|----------------|
| **ConfigMap** | Non-sensitive config in env vars — usually safe to commit |
| **Secret** | Base64 ≠ encryption. Never commit real values. Use `secret.example.yaml` |
| **StatefulSet** | Stable identity (`postgres-0`), `volumeClaimTemplates` (NOT `volumes:` in container) |
| **PVC** | Storage survives Pod restarts — decoupled from Pod lifecycle |
| **Headless Service** | `clusterIP: None` enables direct Pod DNS for StatefulSet |
| **Rolling Update** | Zero-downtime — change image tag, let K8s handle the rest |
| **Mock → Real** | Start with fake JSON, swap to DB later — API contract stays same |

## File Tree

```
07.pv,configmap,secrers/
├── note.md
├── note.ipynb
├── backend/
│   ├── main.py                    (v1.1 with psycopg2, reads env vars)
│   ├── Dockerfile
│   ├── requirements.txt
│   └── backend.yaml               (Deployment + ClusterIP Service)
├── frontend/
│   ├── main.py                    (calls http://backend, returns HTML)
│   ├── Dockerfile
│   ├── requirements.txt
│   └── frontend.yaml              (Deployment + ClusterIP Service)
└── postgres/
    ├── secrets.yaml               (POSTGRES_USER, POSTGRES_PASSWORD)
    ├── pvc.yaml                   (1Gi ReadWriteOnce)
    ├── statefulset.yaml           (Postgres 16, volumeClaimTemplates)
    └── service.yaml               (headless, clusterIP: None)
```

## Final Project
Production deployment will be manual (not committed). Real `secret.yaml` will be generated at deploy time from a secrets manager.